# 🌳 EDA Senior – Category Tree
**Gráficas:** 6 | **Dataset:** `category_tree_clean.csv` / `category_tree.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from src.config import *
from src.data_cleaning import load_category_tree, summarize_dataframe
from src.viz import setup_style
setup_style('ocean')

path = CATEGORY_TREE_CLEAN if os.path.exists(CATEGORY_TREE_CLEAN) else CATEGORY_TREE_RAW
print(f'Cargando: {path}')
df = load_category_tree(path)
summarize_dataframe(df, 'Category Tree')
print('Columnas detectadas:', df.columns.tolist())

In [ ]:
# ── Auto-detectar columnas 'id' y 'parentid' ──
cols = df.columns.tolist()
# Heurística: columna con valores únicos = id; columna con muchos NaN = parentid
null_pct = df.isnull().mean()
parent_col = null_pct.idxmax()          # columna con más nulos → parentid
id_col     = [c for c in cols if c != parent_col][0]
print(f'ID column     : {id_col}')
print(f'Parent column : {parent_col}')

n_total  = len(df)
n_roots  = int(df[parent_col].isnull().sum())
all_ids  = set(df[id_col].dropna())
parent_ids = set(df[parent_col].dropna())
n_leaves = len(all_ids - parent_ids)
n_middle = n_total - n_roots - n_leaves

print(f'\nTotal categorías : {n_total:,}')
print(f'Raíces (sin padre): {n_roots:,}')
print(f'Hojas (sin hijos) : {n_leaves:,}')
print(f'Intermedios       : {n_middle:,}')

## 📊 Gráfica 1 – Composición del Árbol (Pie)

In [ ]:
sizes  = [n_roots, n_leaves, max(n_middle, 0)]
labels = ['Raíces (sin padre)', 'Hojas (sin hijos)', 'Nodos intermedios']
colors = ['#e74c3c','#2ecc71','#3498db']
# Quitar segmentos en 0
data = [(l,s,c) for l,s,c in zip(labels,sizes,colors) if s > 0]
labels_, sizes_, colors_ = zip(*data)

fig, ax = plt.subplots(figsize=(7,7))
ax.pie(sizes_, labels=labels_, autopct='%1.1f%%', colors=colors_,
       wedgeprops=dict(edgecolor='white', linewidth=2), startangle=120)
ax.set_title('Composición del Árbol de Categorías', fontweight='bold')
plt.tight_layout(); plt.show()

## 📊 Gráficas 2-3 – Hijos por Nodo

In [ ]:
children_per_parent = df.groupby(parent_col)[id_col].count().sort_values(ascending=False)

fig, axes = plt.subplots(1,2, figsize=(14,5))
# Gráfica 2: histograma
children_per_parent.value_counts().sort_index().head(25).plot(
    kind='bar', ax=axes[0], color='#9b59b6', edgecolor='white')
axes[0].set_title('Distribución Nº Hijos por Nodo', fontweight='bold')
axes[0].set_xlabel('Nº hijos'); axes[0].set_ylabel('Nº nodos')

# Gráfica 3: top parents
top_p = children_per_parent.head(15)
sns.barplot(x=top_p.values, y=top_p.index.astype(str), palette='plasma', ax=axes[1])
axes[1].set_title('Top 15 Nodos con Más Hijos Directos', fontweight='bold')
axes[1].set_xlabel('Nº hijos'); axes[1].set_ylabel('Parent ID')
plt.tight_layout(); plt.show()

## 📊 Gráficas 4-5 – Profundidad del Árbol (BFS)

In [ ]:
# BFS iterativo para calcular profundidad de cada nodo
parent_map = dict(zip(df[id_col].dropna(), df[parent_col]))
depth_map  = {}
for node in df[id_col].dropna():
    d, current, visited = 0, node, set()
    while pd.notna(parent_map.get(current)) and current not in visited:
        visited.add(current)
        current = parent_map[current]
        d += 1
        if d > 20: break  # seguridad contra ciclos
    depth_map[node] = d

df['depth'] = df[id_col].map(depth_map)
print('Distribución por profundidad:')
print(df['depth'].value_counts().sort_index().to_string())
print(f'\nProfundidad máxima: {int(df["depth"].max())}')
print(f'Profundidad media : {df["depth"].mean():.2f}')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
# Gráfica 4: bar de profundidades
df['depth'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#1abc9c', edgecolor='white')
axes[0].set_title('Categorías por Nivel de Profundidad', fontweight='bold')
axes[0].set_xlabel('Nivel'); axes[0].set_ylabel('Nº categorías')
axes[0].tick_params(axis='x', rotation=0)

# Gráfica 5: barras apiladas acumuladas
level_counts = df['depth'].value_counts().sort_index()
cumulative, cmap = 0, plt.cm.get_cmap('tab10', len(level_counts))
for i, (lvl, cnt) in enumerate(level_counts.items()):
    axes[1].bar('Árbol', cnt, bottom=cumulative, label=f'Nivel {lvl}', color=cmap(i))
    cumulative += cnt
axes[1].set_title('Árbol de Categorías – Vista Apilada por Nivel', fontweight='bold')
axes[1].set_ylabel('Categorías'); axes[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

## 📊 Gráfica 6 – Boxplot Hijos por Nivel

In [ ]:
child_counts = df.groupby(parent_col)[id_col].count().rename('n_children')
df_plot = df.join(child_counts, on=id_col).fillna(0)

fig, ax = plt.subplots(figsize=(12,5))
df_plot.boxplot(column='n_children', by='depth', ax=ax)
ax.set_title('Distribución de Hijos por Nivel del Árbol', fontweight='bold')
ax.set_xlabel('Profundidad'); ax.set_ylabel('Nº hijos directos')
plt.suptitle('')
plt.tight_layout(); plt.show()

---
## ✅ Resumen para Modelado

| Hallazgo | Implicación |
|----------|-------------|
| Árbol multinivel | Usar jerarquía como feature de similaridad |
| Muchas hojas | Target directo para CBF |
| Alta ramificación en algunos nodos | Riesgo de recomendaciones genéricas |
| Profundidad variable | Padding necesario en modelos de grafos |